# Supplementary Table 1

Candidate-clustering search sensitivity in 15 CAMPI acquisitions: for each acquisition, the unclustered search and six MMseqs2/Linclust identity thresholds (99, 97, 95, 90, 75 and 50%), 105 rows. This notebook rebuilds the table from `source_data/STab1/`, compares it with the table as produced, repeats the producer's checks and prints the medians quoted in the Results.

Run from the repository root: `python paper.py build STab1`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

105 rows, one per acquisition (n = 15 CAMPI acquisitions) and arm (unclustered plus six identity thresholds). Supports "all 105 acquisition–arm configurations and paired counts are supplied in Supplementary Table 1" (v28d line 695) and the Results sentence on clustering (v28d line 89). No test; counts and paired ratios relative to the same acquisition's unclustered search.

In [ ]:
import pandas as pd
d = panels.load(SD)
table = panels.build_table(d)
tsv, xlsx = panels.write_table(table, OUT)
print(table.shape)
pd.set_option("display.max_columns", 30)
table.head(14)

## Producer checks

The earlier 45 rows (unclustered, 97%, 99%) and the 60 rows added for 95/90/75/50% must equal the producer table under the producer's column mapping; an independent recount from the Sage result files must agree on all 105 rows; all 90 clustered rows must lie below their unclustered baseline.

In [ ]:
chk, ok = panels.checks(d)
chk.to_csv(OUT / "CHECKS.tsv", sep="\t", index=False)
print(chk.to_string(index=False))
print("ALL_CHECKS_PASS" if ok else "CHECK_FAILURE")
assert ok

## Checks against the table as produced and recomputed numbers

The TSV must be byte-identical to `expected/Supplementary_Table_1.tsv`; the XLSX is compared cell by cell (its bytes carry a timestamp); the check table must be byte-identical to `expected/CHECKS.tsv`. Receipts go to `checks/`. Then every number in `NUMBERS.tsv` is printed in order.

In [ ]:
import hashlib, json
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
exp = UNIT / "expected"
receipt = {
    "tsv": {"new_sha256": sha(tsv), "expected_sha256": sha(exp / "Supplementary_Table_1.tsv")},
    "checks_tsv": {"new_sha256": sha(OUT / "CHECKS.tsv"), "expected_sha256": sha(exp / "CHECKS.tsv")},
    "xlsx": panels.compare_xlsx_values(xlsx, exp / "Supplementary_Table_1.xlsx"),
}
receipt["tsv"]["identical"] = receipt["tsv"]["new_sha256"] == receipt["tsv"]["expected_sha256"]
receipt["checks_tsv"]["identical"] = receipt["checks_tsv"]["new_sha256"] == receipt["checks_tsv"]["expected_sha256"]
receipt["xlsx"]["expected_sha256"] = sha(exp / "Supplementary_Table_1.xlsx")
receipt["result"] = "IDENTICAL" if all(receipt[k]["identical"] for k in ("tsv", "checks_tsv", "xlsx")) else "DIFFERENT"
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "table.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))

med = panels.stat_medians(d)
print("NUMBERS.tsv order")
print("89  acquisitions reduced at every threshold:", panels.stat_acquisitions_reduced_at_every_threshold(d))
for arm in ["cluster_099", "cluster_097", "cluster_095", "cluster_090", "cluster_075", "cluster_050"]:
    print(f"89  median peptide reduction {arm}: {repr(float(med.loc[arm, 'red']))} (n={int(med.loc[arm, 'n'])})")
lo, hi = panels.stat_ko_median_range_pct(d)
print("89  lowest median KO recovery %:", repr(lo))
print("89  highest median KO recovery %:", repr(hi))
print("475 acquisitions:", panels.stat_n_acquisitions(d))
print("695 acquisition-arm configurations:", panels.stat_n_rows(d))
print("(check) clustered rows below baseline:", panels.stat_rows_below_baseline(d))